# 3. Preprocesamiento y particiones

Reglas del curso (capítulo 10 y sección 9.10.4.2):

1. La partición se define **una sola vez, antes de transformar nada**, y se guarda en disco
   (`particion_p*.parquet` con `id` y `split`). Todos los modelos la leen.
2. Partición **temporal**: con datos en el tiempo, mezclar el futuro en el entrenamiento es fuga de
   información.
3. Imputación, escalado y codificación van dentro de un `Pipeline` con `ColumnTransformer`, ajustados
   solo con entrenamiento (en cada pliegue de la validación cruzada).
4. Validación cruzada temporal (`TimeSeriesSplit` por periodo) dentro de `GridSearchCV`.

In [1]:
import sys, warnings, json, time
from pathlib import Path
ML = Path.cwd().parent                      # carpeta ml/
sys.path.insert(0, str(ML))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
PROCESSED, REPORTS = ML / "data" / "processed", ML / "reports"
PRED = REPORTS / "predicciones"; PRED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", palette="deep")

In [2]:
# ===================== CONFIGURACIÓN =====================
P1_MESES_PRUEBA = 12        # últimos meses reservados para prueba
P2_ULTIMO_ANIO_ENTRENAMIENTO = 2023   # años objetivo posteriores van a prueba
P3_MESES_PRUEBA = 36        # origen móvil en los últimos 36 meses
N_PLIEGUES_CV = 4           # pliegues de la validación cruzada temporal

In [3]:
P1 = pd.read_parquet(PROCESSED / "dataset_p1.parquet"); P2 = pd.read_parquet(PROCESSED / "dataset_p2.parquet")
P3 = pd.read_parquet(PROCESSED / "dataset_p3.parquet")
meses = np.sort(P1["mes"].unique()); corte = meses[-P1_MESES_PRUEBA]
P1["split"] = np.where(P1["mes"] < corte, "train", "test")
P2["split"] = np.where(P2["anio_objetivo"] <= P2_ULTIMO_ANIO_ENTRENAMIENTO, "train", "test")
for nombre, d in (("p1", P1), ("p2", P2)):
    d[["id", "split"]].to_parquet(PROCESSED / f"particion_{nombre}.parquet", index=False)
    print(nombre, d["split"].value_counts().to_dict(), "| ids únicos:", d["id"].is_unique)
print("P1 prueba:", pd.Timestamp(corte).date(), "→", P1.mes.max().date())
print("P3 prueba (origen móvil):", P3.index[-P3_MESES_PRUEBA].date(), "→", P3.index[-1].date())
json.dump({"P1_MESES_PRUEBA": P1_MESES_PRUEBA, "P2_ULTIMO_ANIO_ENTRENAMIENTO": P2_ULTIMO_ANIO_ENTRENAMIENTO,
           "P3_MESES_PRUEBA": P3_MESES_PRUEBA, "N_PLIEGUES_CV": N_PLIEGUES_CV}, open(PROCESSED / "configuracion_particion.json", "w"))

p1 {'train': 457, 'test': 234} | ids únicos: True
p2 {'train': 46477, 'test': 19262} | ids únicos: True
P1 prueba: 2025-07-01 → 2026-06-01
P3 prueba (origen móvil): 2023-08-31 → 2026-07-31


## 3.1 Pipeline de preprocesamiento

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
def preprocesador(num, cat):
    numerico = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
    categorico = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                           ("codificar", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False))])
    return ColumnTransformer([("num", numerico, num)] + ([("cat", categorico, cat)] if cat else []))
preprocesador(["log_activos", "de"], ["ciiu_division"])

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputar',
                                                  SimpleImputer(strategy='median')),
                                                 ('escalar',
                                                  StandardScaler())]),
                                 ['log_activos', 'de']),
                                ('cat',
                                 Pipeline(steps=[('imputar',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('codificar',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=20,
                                                                sparse_output=False))]),
                                 ['ciiu_division'])])

## 3.2 Validación cruzada temporal

In [5]:
from sklearn.model_selection import TimeSeriesSplit
def pliegues_temporales(periodos, n_splits=N_PLIEGUES_CV):
    u = np.sort(pd.Series(periodos).unique())
    return [(np.where(pd.Series(periodos).isin(u[a]))[0], np.where(pd.Series(periodos).isin(u[b]))[0])
            for a, b in TimeSeriesSplit(n_splits=min(n_splits, len(u) - 1)).split(u)]
a = P2.loc[P2.split == "train", "anio_objetivo"].reset_index(drop=True)
for i, (tr, va) in enumerate(pliegues_temporales(a)):
    print(f"P2 pliegue {i+1}: entrena {sorted(a.iloc[tr].unique())} → valida {sorted(a.iloc[va].unique())}")

P2 pliegue 1: entrena [2017, 2018, 2019] → valida [2020]
P2 pliegue 2: entrena [2017, 2018, 2019, 2020] → valida [2021]
P2 pliegue 3: entrena [2017, 2018, 2019, 2020, 2021] → valida [2022]
P2 pliegue 4: entrena [2017, 2018, 2019, 2020, 2021, 2022] → valida [2023]


## 3.3 Transformaciones de los objetivos de P2

| Objetivo | Transformación | Motivo |
|---|---|---|
| D/E | log(1 + D/E), acotado en 20 | Distribución muy sesgada |
| Cobertura | asinh, acotada en ±50; sin intereses = 0 | Valores extremos y negativos |
| Kd contable | Solo valores entre 0 % y 60 % | Elimina errores de registro |
| ROA | Acotado en ±1 | Elimina errores de registro |